# 03 — Country-Year Exposure + Mundlak Decomposition

**Day 3, Track A.** Build the level-2 exposure variables $\bar E_{ct}$, $\bar E_c$, and $E_{ct} - \bar E_c$ from the per-individual ESS panel + ILO–NASK scores merged in notebook 02.

## What this notebook does

1. Reads `data/interim/ess_panel_with_scores.parquet`.
2. Aggregates `genai_i` to a (country × round) cell using ESS design weights `pspwght`. Same for `genai_i_static`.
3. Applies the Mundlak / within-between decomposition: country-mean (between) + within-deviation columns.
4. Verifies the identity `exposure_ct == exposure_within + exposure_between` element-wise (Day-3 hard checkpoint).
5. Computes the **leave-one-out** country-year mean for the §7 robustness check.
6. Joins the country-year and decomposition columns back to the individual panel.
7. Persists `data/interim/country_year_exposure.parquet` (the L2 frame) and `data/interim/ess_panel_with_l2.parquet` (panel + L2 columns).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mla.mundlak import (  # noqa: E402
    country_year_aggregate,
    country_year_aggregate_leave_one_out,
    merge_country_year_to_panel,
    within_between_decompose,
)

INTERIM_DIR = REPO_ROOT / "data" / "interim"
REPO_ROOT

PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA')

## 1. Load the merged panel with scores

In [2]:
panel = pd.read_parquet(INTERIM_DIR / "ess_panel_with_scores.parquet")
print(f"shape: {panel.shape}")
print(f"columns include genai: {[c for c in panel.columns if 'genai' in c]}")

shape: (276491, 23)
columns include genai: ['genai_i', 'genai_i_static']


## 2. Country-year aggregation (weighted by `pspwght`)

In [3]:
cy = country_year_aggregate(
    panel, "genai_i", weight_col="pspwght", out_col="exposure_ct"
)
cy_static = country_year_aggregate(
    panel, "genai_i_static", weight_col="pspwght", out_col="exposure_ct_static"
)
cy = cy.merge(cy_static, on=["cntry", "essround"], how="outer")
print(f"L2 frame: {cy.shape}  (countries × rounds)")
print(f"unique countries: {cy['cntry'].nunique()}")
print(f"unique rounds:    {sorted(cy['essround'].unique().tolist())}")
cy.head()

L2 frame: (154, 4)  (countries × rounds)
unique countries: 36
unique rounds:    [6, 7, 8, 9, 10, 11]


,cntry,essround,exposure_ct,exposure_ct_static
0,AL,6,0.274885,0.266540
1,AT,7,0.335501,0.320404
2,AT,8,0.327182,0.307046
3,AT,9,0.332505,0.320229
4,AT,11,0.329899,0.329899


In [4]:
cy.describe().round(4)

,essround,exposure_ct,exposure_ct_static
count,154.0,154.0000,154.0000
mean,8.5455,0.3085,0.3007
std,1.7679,0.0216,0.0205
min,6.0,0.2557,0.2492
25%,7.0,0.2936,0.2872
50%,9.0,0.3086,0.3006
75%,10.0,0.3243,0.3170
max,11.0,0.3528,0.3465


## 3. Mundlak / within-between decomposition

Adds:
* `exposure_ct_between` = $\bar E_c$ (country mean across rounds)
* `exposure_ct_within`  = $E_{ct} - \bar E_c$ (within-country deviation)

And the static-vintage analogues for the §7 robustness comparator.

In [5]:
decomp = within_between_decompose(cy, "exposure_ct")
decomp = within_between_decompose(decomp, "exposure_ct_static")
decomp.head()

,cntry,essround,exposure_ct,exposure_ct_static,exposure_ct_between,exposure_ct_within,exposure_ct_static_between,exposure_ct_static_within
0,AL,6,0.274885,0.266540,0.274885,0.000000,0.266540,0.000000
1,AT,7,0.335501,0.320404,0.331272,0.004229,0.319394,0.001010
2,AT,8,0.327182,0.307046,0.331272,-0.004090,0.319394,-0.012348
3,AT,9,0.332505,0.320229,0.331272,0.001233,0.319394,0.000834
4,AT,11,0.329899,0.329899,0.331272,-0.001373,0.319394,0.010505


### Identity check (Day-3 hard checkpoint)

In [6]:
identity_gap = (
    decomp["exposure_ct"]
    - decomp["exposure_ct_between"]
    - decomp["exposure_ct_within"]
).abs().max()
static_identity_gap = (
    decomp["exposure_ct_static"]
    - decomp["exposure_ct_static_between"]
    - decomp["exposure_ct_static_within"]
).abs().max()
print(f"max |E_ct - between - within|:        {identity_gap:.2e}")
print(f"max |E_ct_static - between - within|: {static_identity_gap:.2e}")
checkpoint = max(identity_gap, static_identity_gap) < 1e-12
print(f"Day-3 identity checkpoint: {'PASS' if checkpoint else 'FAIL'}")

max |E_ct - between - within|:        0.00e+00
max |E_ct_static - between - within|: 0.00e+00
Day-3 identity checkpoint: PASS


## 4. Within vs static-vintage spread

How much of the within-country variation is the R10→R11 vintage shock vs. compositional drift in $w_{oct}$? Compare the SD of `exposure_ct_within` (vintage) to the SD of `exposure_ct_static_within` (composition only).

In [7]:
spread = pd.DataFrame(
    {
        "sd_within_vintage": [decomp["exposure_ct_within"].std()],
        "sd_within_static": [decomp["exposure_ct_static_within"].std()],
    },
    index=["all_rounds"],
).round(5)
spread["vintage_excess"] = spread["sd_within_vintage"] - spread["sd_within_static"]
spread

,sd_within_vintage,sd_within_static,vintage_excess
all_rounds,0.00818,0.00787,0.00031


In [8]:
# Same comparison restricted to non-R11 rows (where vintage = static = 2023):
# the within-SDs should be identical here.
non_r11 = decomp[decomp["essround"] != 11]
print(f"sd within (vintage)  R6–R10 only: {non_r11['exposure_ct_within'].std():.5f}")
print(f"sd within (static)   R6–R10 only: {non_r11['exposure_ct_static_within'].std():.5f}")

sd within (vintage)  R6–R10 only: 0.00773
sd within (static)   R6–R10 only: 0.00742


## 5. Leave-one-out country-year exposure (§7 robustness check)

In [9]:
loo = country_year_aggregate_leave_one_out(
    panel, "genai_i", weight_col="pspwght", out_col="exposure_ct_loo"
)
panel_loo = panel.copy()
panel_loo["exposure_ct_loo"] = loo
print(f"non-null LOO exposure: {loo.notna().sum():,} of {len(loo):,}")
# Compare to the standard country-year mean for the same row.
panel_loo = panel_loo.merge(
    decomp[["cntry", "essround", "exposure_ct"]],
    on=["cntry", "essround"], how="left"
)
diff = (panel_loo["exposure_ct_loo"] - panel_loo["exposure_ct"]).abs()
print(f"|loo - mean| 99th percentile: {diff.quantile(0.99):.5f}")
print(f"|loo - mean| max:             {diff.max():.5f}")

non-null LOO exposure: 230,473 of 276,491
|loo - mean| 99th percentile: 0.00044
|loo - mean| max:             0.00334


## 6. Join country-year columns back to individual panel

In [10]:
panel_with_l2 = merge_country_year_to_panel(
    panel,
    decomp,
    cy_value_cols=[
        "exposure_ct",
        "exposure_ct_between",
        "exposure_ct_within",
        "exposure_ct_static",
        "exposure_ct_static_between",
        "exposure_ct_static_within",
    ],
)
panel_with_l2["exposure_ct_loo"] = loo
print(f"final panel shape: {panel_with_l2.shape}")
panel_with_l2[
    ["cntry", "essround", "genai_i", "exposure_ct",
     "exposure_ct_between", "exposure_ct_within"]
].head()

final panel shape: (276491, 30)


,cntry,essround,genai_i,exposure_ct,exposure_ct_between,exposure_ct_within
0,AL,6,0.17,0.274885,0.274885,0.0
1,AL,6,NaN,0.274885,0.274885,0.0
2,AL,6,0.36,0.274885,0.274885,0.0
3,AL,6,NaN,0.274885,0.274885,0.0
4,AL,6,0.41,0.274885,0.274885,0.0


## 7. Persist outputs

In [11]:
cy_out = INTERIM_DIR / "country_year_exposure.parquet"
panel_out = INTERIM_DIR / "ess_panel_with_l2.parquet"
decomp.to_parquet(cy_out, index=False)
panel_with_l2.to_parquet(panel_out, index=False)
print(f"wrote {cy_out}    ({cy_out.stat().st_size/1e6:.2f} MB)")
print(f"wrote {panel_out} ({panel_out.stat().st_size/1e6:.2f} MB)")

wrote /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/interim/country_year_exposure.parquet    (0.01 MB)
wrote /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/interim/ess_panel_with_l2.parquet (6.71 MB)
